# 6. A complete regression workflow
## Goal
Compare a trivial baseline, a regularized linear model, and a tree ensemble using training-side cross-validation, then evaluate the selected model once on held-back data.
Course Modules 9–10. This uses generated independent observations and a deliberately simple regression relationship, with 5% missing feature entries. The target uses arbitrary units. Scores are teaching results, not forecasts of real-world model performance.
## Setup
Uses NumPy and scikit-learn; requires no downloads or credentials.
References: [scikit-learn pitfalls](https://scikit-learn.org/stable/common_pitfalls.html), [Inria course](https://inria.github.io/scikit-learn-mooc/).
## Steps
### 1. Generate data and reserve a test set
The split is random because the generated observations are independent. This choice would need reconsideration for time series or grouped records.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

X, y = make_regression(n_samples=700, n_features=5, n_informative=4, noise=20, random_state=42)
rng = np.random.default_rng(42)
X[rng.random(X.shape) < 0.05] = np.nan
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train and test shapes:", X_train.shape, X_test.shape)
assert len(X_train) == 560 and len(X_test) == 140

Train and test shapes: (560, 5) (140, 5)


### 2. Define candidates and the selection rule
Choose lowest mean training-side CV MAE. Each fold learns its own imputer/scaler. The held-back test is not used here.

In [2]:
candidates = {
    "dummy median": make_pipeline(SimpleImputer(strategy="median"), DummyRegressor(strategy="median")),
    "ridge": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=1.0)),
    "random forest": make_pipeline(SimpleImputer(strategy="median"), RandomForestRegressor(n_estimators=100, min_samples_leaf=3, random_state=42, n_jobs=1)),
}
cv = KFold(n_splits=5, shuffle=True, random_state=42)
rows = []
for name, model in candidates.items():
    errors = -cross_val_score(model, X_train, y_train, cv=cv, scoring="neg_mean_absolute_error", n_jobs=1)
    rows.append({"model": name, "cv_mae": errors.mean(), "fold_sd": errors.std(ddof=1)})
comparison = pd.DataFrame(rows).sort_values("cv_mae").reset_index(drop=True)
print(comparison.to_string(index=False))
selected_name = comparison.iloc[0]["model"]
print("Selected using training data only:", selected_name)

        model     cv_mae   fold_sd
        ridge  25.035324  2.582208
random forest  36.436683  4.054826
 dummy median 109.541779 10.277053
Selected using training data only: ridge


### 3. Freeze the decision and evaluate
Both the selected model and reference baseline are fitted on all training data. Fold SD above is descriptive variability, not a confidence interval.

In [3]:
selected_model = candidates[selected_name]
selected_model.fit(X_train, y_train)
predictions = selected_model.predict(X_test)
dummy = make_pipeline(SimpleImputer(strategy="median"), DummyRegressor(strategy="median"))
dummy.fit(X_train, y_train)
test_mae = mean_absolute_error(y_test, predictions)
baseline_mae = mean_absolute_error(y_test, dummy.predict(X_test))
print(f"Final test MAE: {test_mae:.2f} target units")
print(f"Baseline test MAE: {baseline_mae:.2f} target units")
print(f"Relative MAE reduction: {(baseline_mae-test_mae)/baseline_mae:.1%}")
assert predictions.shape == y_test.shape
assert np.isfinite(predictions).all()
assert test_mae >= 0 and baseline_mae >= 0

Final test MAE: 25.70 target units
Baseline test MAE: 107.39 target units
Relative MAE reduction: 76.1%


### 4. Inspect errors after final assessment
Use these examples to understand limitations. If they inspire model changes, you need fresh evaluation evidence for the revised model.

In [4]:
errors = pd.DataFrame({"actual": y_test, "predicted": predictions})
errors["absolute_error"] = (errors["actual"] - errors["predicted"]).abs()
print(errors.nlargest(5, "absolute_error").round(2).to_string(index=False))

 actual  predicted  absolute_error
 243.60      60.82          182.77
-231.77     -60.45          171.32
 148.36      13.62          134.73
 176.91      62.54          114.37
 -79.97      19.67           99.64


## Checks
Explain why imputation lives inside the pipeline. Explain why a linear model can do well on this generated relationship and why this does not rank model families universally. Report the actual outputs, including a failure to improve if one occurs; do not make passing conditional on beating a baseline on every dataset.
## Next Steps
Exercises 30–35. For a new experiment, work only with training-side validation and record choices before execution. The existing test has now been revealed: do not claim it remains a pristine test for an adaptively redesigned model. Your independent capstone needs its own declared split.